# 02 · High-ticket machines

This notebook replicates the High-ticket machines paragraph of the report's Results section.

> High-ticket machines: We found 113 chronic outliers at 82 customers. The top 5% of the 6,621 telemetry devices hold 32.9% of their tickets vs. 22.2% by chance. Allowing for spread between machines, only 2 rate outliers still stand out, so high-ticket machines form a continuum. Non-success codes rise before their tickets no more than on other machines.

We also check the numbers behind two of its statements: the 118 rate outliers that "only 2" refers to (`docs/05_high_ticket_machines.md`), and the rise in non-success codes in each group, which the report's Definitions give as +20.3 for chronic outliers, +21.6 for other top-5% machines, and +21.5 for all other machines.

**How it works.** `high_ticket_machines.py` links IPA's case export to the clearView telemetry devices, gives each device an expected ticket count (its telemetry type's rate × its months active), flags rate outliers and chronic outliers, compares ticket concentration with 1,000 Poisson simulations, re-tests the rate outliers with a negative binomial, and summarizes non-success codes before each ticket. We recompute every number from the tables it writes, with one row per machine, device, or ticket. We also rebuild these derived columns from the telemetry-database extracts or from their definitions and check them against the script's: each device's months active, tickets, expected tickets, and p-values; each chronic outlier's customer; and each ticket's group and comparison window. For the chance benchmark, we redraw the script's 1,000 simulations with the seed recorded in `htm_summary.json` and check our median against the one it records.

**Re-run.** With `RERUN = True`, we re-run `verification/high_ticket_machines.py`, which takes about 15 s. It rewrites its 13 `htm_*.csv` tables and `htm_summary.json` in a scratch copy of `verification/outputs/`, without its figures, and writes its two real-name tables to a scratch copy of `data/private/`, which the notebook does not read and which is deleted when the kernel exits. We then compare each of the 14 output files with the committed one.

**Used as committed, without re-running.** The script also reads four extracts of the telemetry database: `fu/fu_ticket_last10.csv` (each ticket's device and its 30-day windows), `fu/fu_devices.csv` (the device dictionary), `db/db_host_level.csv` (which host is IPA's shared internal host), and `db/db_machine_type_rates.csv` (the midterm analysis's rates by telemetry type, for comparison only). `db_followup.py` and `verify_with_db.py` wrote them from the telemetry database, which the notebooks do not re-read, so the re-run uses the committed files. This notebook reads the first three as well. It also reads `followups/rp_pairs.csv`, a table of `followups.py` with one row per eligible ticket and its account; notebook 03 re-runs that script.

In [1]:
RERUN = True  # False: recompute from the committed outputs without re-running the scripts
import pandas as pd
import replicate as R
OUT = R.outputs(RERUN)

RERUN is True: the scripts write into a scratch copy of verification/outputs/.


In [2]:
if RERUN:
    R.rerun(OUT, 'high_ticket_machines.py')
R.compare(OUT)

high_ticket_machines.py: exit status 0, 15 s
    Done in 14.2 s; checks {'PASS': 33, 'FAIL': 0}
14 files written: 14 identical.


,file written by the re-run,compared with the committed file
0,htm_chronic_categories.csv,identical
1,htm_chronic_top3.csv,identical
2,htm_customer_chronic.csv,identical
3,htm_device_exposure.csv,identical
4,htm_episodes.csv,identical
5,htm_issue_mix.csv,identical
6,htm_lorenz.csv,identical
7,htm_machine_counts.csv,identical
8,htm_pre_ticket.csv,identical
9,htm_pre_ticket_summary.csv,identical


## The 6,621 telemetry devices

The exposure population is every device in the telemetry device table except the devices on IPA's shared internal host and the devices with no host. We rebuild it from `fu/fu_devices.csv` and `db/db_host_level.csv` and check that it is the set of devices in `htm_device_exposure.csv`, the script's table with one row per device. From `fu/fu_devices.csv`, we also rebuild each device's months active: the calendar months from its first to its last transaction between Aug 1st, 2025, and Jul 31st, 2026. From `fu/fu_ticket_last10.csv`, we count each device's unplanned tickets (every ticket except planned work, canceled requests, and duplicates) opened in those 12 months. We check both columns against the table.

In [3]:
devices = R.read_csv(OUT, 'fu/fu_devices.csv')  # one row per telemetry device (database extract)
hosts = R.read_csv(OUT, 'db/db_host_level.csv')
windows = R.read_csv(OUT, 'fu/fu_ticket_last10.csv', low_memory=False)  # four windows per device-matched ticket
dev_exp = R.read_csv(OUT, 'htm_device_exposure.csv')  # one row per device in the exposure population

on_shared_host = devices.host.isin(hosts.host[hosts.shared_internal_host == 1])
no_host = devices.host.isna()
population = devices.machine_id[~on_shared_host & ~no_host]
assert dev_exp.machine_id.is_unique and set(dev_exp.machine_id) == set(population), \
    'htm_device_exposure.csv is not the rebuilt population'
n_devices = len(population)

# Months active: the calendar months from the first to the last transaction day inside the 12 months
first = pd.to_datetime(devices.first_active).clip(lower=pd.Timestamp('2025-08-01'))
last = pd.to_datetime(devices.last_active).clip(upper=pd.Timestamp('2026-07-31'))
months_active = (12 * (last.dt.year - first.dt.year) + last.dt.month - first.dt.month + 1).where(first <= last, 0)
# Unplanned tickets opened in the 12 months, counted on one of the four windows of each ticket
matched = windows[windows.window == 'corrected_guard0']  # one row per device-matched ticket: the 30 days before it
unplanned = matched[~matched.service_group.isin(['Planned work', 'Cancelled request', 'Duplicate case'])]
in_window = unplanned.opened_at_eastern.str[:10].between('2025-08-01', '2026-07-31')
tickets_in_window = unplanned.machine_id[in_window].value_counts()
rebuilt = pd.DataFrame({'months': months_active.astype(int).to_numpy(),
                        'tickets_obs': devices.machine_id.map(tickets_in_window).fillna(0).astype(int).to_numpy()},
                       index=devices.machine_id).reindex(dev_exp.machine_id)
assert (rebuilt.to_numpy() == dev_exp[['months', 'tickets_obs']].to_numpy()).all(), \
    'the months active or tickets differ from htm_device_exposure.csv'
print(f'Rebuilt from the extracts, the {n_devices:,} devices have {rebuilt.months.sum():,} months active and '
      f'{rebuilt.tickets_obs.sum():,} unplanned tickets, device by device as in htm_device_exposure.csv.')
pd.Series({'telemetry devices': len(devices), 'on the shared internal host': int(on_shared_host.sum()),
           'with no host': int(no_host.sum()), 'exposure population': n_devices}, name='devices').to_frame()

Rebuilt from the extracts, the 6,621 devices have 71,559 months active and 5,395 unplanned tickets, device by device as in htm_device_exposure.csv.


,devices
telemetry devices,7183
on the shared internal host,411
with no host,151
exposure population,6621


## 113 chronic outliers at 82 customers

A chronic outlier is a device in the top 5% of machines by unplanned tickets that is also a rate outlier. We take the top-5% cut from `htm_machine_counts.csv` (one row per ticketed machine: the count at rank 185, ties included).

A rate outlier has improbably many tickets for its expected tickets, its type's rate × its months active. A type's rate is its devices' tickets ÷ their months active, and the types with under 100 months active share one pooled rate. For each device with expected tickets above 0, we compute the one-sided Poisson p-value (the chance of at least its tickets) and the Benjamini–Hochberg q-value over those devices. A rate outlier has q < 0.05. We check the rate groups, the p-values, and the rate outliers against `htm_device_exposure.csv`.

The report's customer for a chronic outlier is the account named most often on its tickets, which `fu/fu_devices.csv` records as `ticket_customer`. We count customers from it and check that it agrees with the `customer` column of `htm_device_exposure.csv`. That column holds the account of the device's location when all of the location's ticketed devices have the same `ticket_customer`, and the device's own `ticket_customer` otherwise. As an independent check, we take the account on each of the chronic outliers' eligible tickets (the unplanned tickets opened before Aug 2nd, 2026) from `followups/rp_pairs.csv` and check that the account named most often is `ticket_customer` for every chronic outlier.

In [4]:
import math
import numpy as np


def bh(p):
    """Benjamini–Hochberg q-values, computed as in high_ticket_machines.py."""
    p = np.asarray(p, float)
    order = np.argsort(p, kind='mergesort')
    q = np.minimum.accumulate((p[order] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
    out = np.empty(len(p))
    out[order] = np.minimum(q, 1.0)
    return out


def upper_tail(k, mu, alpha=0.0):
    """P(X >= k) for a Poisson with mean mu (alpha = 0) or a negative binomial with mean mu and variance
    mu + alpha mu^2."""
    if alpha > 0:
        size = 1 / alpha

        def log_pmf(j):
            return (math.lgamma(j + size) - math.lgamma(size) - math.lgamma(j + 1)
                    + size * math.log(size / (size + mu)) + j * math.log(mu / (size + mu)))
    else:
        def log_pmf(j):
            return -mu + j * math.log(mu) - math.lgamma(j + 1)
    if k <= mu:  # a large tail: one minus the chance of fewer than k
        return 1 - sum(math.exp(log_pmf(j)) for j in range(k))
    logs = [log_pmf(k)]  # a small tail: the terms fall from k on, so we add them until they are negligible
    while logs[-1] > logs[0] - 50:
        logs.append(log_pmf(k + len(logs)))
    return math.exp(logs[0]) * sum(math.exp(x - logs[0]) for x in logs)


machines = R.read_csv(OUT, 'htm_machine_counts.csv')  # one row per machine with an unplanned ticket
rank_k = math.ceil(0.05 * len(machines))
cut = machines.unplanned_tickets.sort_values(ascending=False).iloc[rank_k - 1]
top5_machines = machines.machine_id[machines.unplanned_tickets >= cut]
top5 = dev_exp.serial_id.isin(top5_machines)

# Expected tickets: the type's rate x the device's months active. Types under 100 months share one pooled rate.
type_months = dev_exp.groupby('machine_type').months.sum()
rate_group = dev_exp.machine_type.where(dev_exp.machine_type.map(type_months) >= 100, '(pooled small types)')
assert (rate_group == dev_exp.rate_group).all(), 'the rate groups differ from htm_device_exposure.csv'
totals = dev_exp.groupby(rate_group)[['tickets_obs', 'months']].sum()
expected = rate_group.map(totals.tickets_obs / totals.months) * dev_exp.months

tested = expected > 0  # the script tests the devices with expected tickets above 0
y, mu = dev_exp.tickets_obs[tested], expected[tested]
p_poisson = pd.Series([upper_tail(k, m) for k, m in zip(y, mu)], index=mu.index)
q_poisson = pd.Series(np.nan, index=dev_exp.index)
q_poisson[tested] = bh(p_poisson)
rate_outlier = q_poisson < 0.05
assert np.allclose(p_poisson, dev_exp.p_poisson[tested], rtol=1e-9, atol=0), \
    "the Poisson p-values differ from the script's"
assert (rate_outlier == (dev_exp.q_bh < 0.05)).all(), 'the rate outliers differ from htm_device_exposure.csv'

chronic = dev_exp[top5 & rate_outlier]
n_chronic = len(chronic)
ticket_customer = chronic.machine_id.map(devices.set_index('machine_id').ticket_customer)
n_chronic_customers = ticket_customer.nunique()
assert (chronic.customer == ticket_customer).all(), "htm_device_exposure.csv's customer is not ticket_customer"
# Independent check: the accounts on the chronic outliers' eligible tickets, one row per ticket (followups.py)
eligible = R.read_csv(OUT, 'followups/rp_pairs.csv')
named_most = (eligible[eligible.machine_id.isin(chronic.serial_id)].groupby('machine_id').customer
              .agg(lambda s: ' and '.join(s.mode())))  # a tie joins two or more accounts
assert (chronic.serial_id.map(named_most) == ticket_customer).all(), \
    "the account named most often on a chronic outlier's eligible tickets is not its ticket_customer"

print(f'Top 5% by unplanned tickets: {cut} or more (the count at rank {rank_k} of {len(machines):,} machines), '
      f'{len(top5_machines)} machines with ties.')
print(f"Our rate groups, Poisson p-values (largest relative difference from the script's: "
      f"{(p_poisson / dev_exp.p_poisson[tested] - 1).abs().max():.0e}), and rate outliers match the script's.")
print(f'{n_chronic} chronic outliers at {n_chronic_customers} customers.')
classes = pd.crosstab(top5.map({True: 'top 5% by unplanned tickets', False: 'other devices'}),
                      rate_outlier.map({True: 'rate outlier', False: 'not a rate outlier'}),
                      rownames=[None], colnames=[None], margins=True, margins_name='all')
classes.loc[['top 5% by unplanned tickets', 'other devices', 'all'], ['rate outlier', 'not a rate outlier', 'all']]

Top 5% by unplanned tickets: 5 or more (the count at rank 185 of 3,685 machines), 247 machines with ties.
Our rate groups, Poisson p-values (largest relative difference from the script's: 5e-13), and rate outliers match the script's.
113 chronic outliers at 82 customers.


,rate outlier,not a rate outlier,all
top 5% by unplanned tickets,113,93,206
other devices,5,6410,6415
all,118,6503,6621


## The top 5% of the devices hold 32.9% of their tickets

We rank the 6,621 devices by their unplanned tickets from Aug 1st, 2025, to Jul 31st, 2026 (`tickets_obs` in `htm_device_exposure.csv`), take the top 5%, and divide their tickets by the devices' total. The top 5% is 5% of the devices, rounded up to a whole number of devices. Because the 5% defines the group, the check table leaves it out.

In [5]:
tickets = dev_exp.tickets_obs.sort_values(ascending=False)
top_k = math.ceil(0.05 * n_devices)
top_share = tickets.iloc[:top_k].sum() / tickets.sum()
print(f'The top {top_k} of {n_devices:,} devices ({R.pct(top_k / n_devices, 1)}) hold {tickets.iloc[:top_k].sum():,} '
      f'of their {tickets.sum():,} tickets: {R.pct(top_share, 1)}.')

The top 332 of 6,621 devices (5.0%) hold 1,774 of their 5,395 tickets: 32.9%.


## 22.2% by chance

The script draws every device's tickets 1,000 times from a Poisson distribution with the device's expected tickets as the mean and takes the median top-5% share over the draws. We redraw the simulations with the seed recorded in `htm_summary.json` and the expected tickets from above, in the device order of `fu/fu_devices.csv` that the draws depend on. We then check our median, our 95th percentile, and the observed share against the values in `htm_summary.json`, which keeps 4 decimals.

In [6]:
summary = R.read_json(OUT, 'htm_summary.json')
d4 = summary['D4_concentration']
means = expected.set_axis(dev_exp.machine_id).reindex(population).to_numpy()  # the draws depend on this order
draws = np.random.default_rng(summary['seed']).poisson(means, size=(summary['n_sim'], n_devices))
top_draws = -np.sort(-draws, axis=1)[:, :top_k].sum(axis=1) / draws.sum(axis=1)
chance_median, chance_p95 = float(np.median(top_draws)), float(np.percentile(top_draws, 95))
ours = {'top5_share_obs': top_share, 'top5_share_null_median': chance_median, 'top5_share_null_p95': chance_p95}
assert all(round(value, 4) == d4[key] for key, value in ours.items()), \
    'the observed or simulated shares differ from htm_summary.json'
pd.DataFrame({'observed': [d4['top5_share_obs'], top_share],
              'chance: median': [d4['top5_share_null_median'], chance_median],
              'chance: 95th percentile': [d4['top5_share_null_p95'], chance_p95]},
             index=['htm_summary.json', 'recomputed here']).map(lambda x: R.pct(x, 2))

,observed,chance: median,chance: 95th percentile
htm_summary.json,32.88%,22.24%,22.81%
recomputed here,32.88%,22.24%,22.81%


## Only 2 of the 118 rate outliers still stand out under a negative binomial

The negative binomial allows each type's ticket rate to vary between machines: a device with expected tickets μ has a ticket count with variance μ + αμ². The script fits the spread α as `alpha_nb` for each rate group. Each telemetry type with at least 100 months active has its own, and the smaller types that make up the pooled rate have one between them. We fit it the same way, by the method of moments over the group's devices with expected tickets above 0: α = Σ((y − μ)² − μ) ÷ Σμ² for tickets y, or 0 if that is negative. We then compute each device's negative-binomial p-value, apply the same Benjamini–Hochberg adjustment over the same devices, check the p-values and the flags against `htm_device_exposure.csv`, and count the rate outliers that keep q < 0.05.

In [7]:
fit = pd.DataFrame({'sq': (y - mu) ** 2 - mu, 'e2': mu ** 2}).groupby(rate_group[tested]).sum()
alpha = rate_group[tested].map((fit.sq / fit.e2).clip(lower=0))
p_nb = pd.Series([upper_tail(k, m, a) for k, m, a in zip(y, mu, alpha)], index=mu.index)
q_nb = pd.Series(np.nan, index=dev_exp.index)
q_nb[tested] = bh(p_nb)
assert np.allclose(p_nb, dev_exp.p_nb[tested], rtol=1e-9, atol=0), \
    "the negative-binomial p-values differ from the script's"
assert ((q_nb < 0.05) == (dev_exp.q_nb < 0.05)).all(), \
    'the negative-binomial outliers differ from htm_device_exposure.csv'
nb_kept = rate_outlier & (q_nb < 0.05)
n_rate_outliers, n_nb_kept = int(rate_outlier.sum()), int(nb_kept.sum())
print(f"{len(fit)} rate groups. Our negative-binomial p-values (largest relative difference from the script's: "
      f"{(p_nb / dev_exp.p_nb[tested] - 1).abs().max():.0e}) and flags match the script's.")
rows = {'rate outliers (Poisson)': rate_outlier, 'still outliers under the negative binomial': nb_kept}
pd.DataFrame([{'devices': int(m.sum()), 'tickets': int(dev_exp.tickets_obs[m].sum()),
               'expected tickets': round(float(expected[m].sum()), 1),
               'median months active': float(dev_exp.months[m].median())} for m in rows.values()],
             index=list(rows))

20 rate groups. Our negative-binomial p-values (largest relative difference from the script's: 7e-14) and flags match the script's.


,devices,tickets,expected tickets,median months active
rate outliers (Poisson),118,837,98.9,12.0
still outliers under the negative binomial,2,24,0.7,4.0


## Non-success codes rise no more before chronic outliers' tickets

`htm_pre_ticket.csv` has one row for each usable unplanned ticket: a device-matched ticket whose 30-day window lies inside the telemetry period and does not start before the machine's first transaction. Each row has the transactions and non-success codes in the 30 days before the ticket and, for a paired ticket, in its comparison window, the same machine's 30 days ending 35 days before the ticket. A paired ticket's comparison window is clean, with no other ticket on the machine 5 to 67 days before the ticket, and it also lies inside the telemetry period and does not start before the machine's first transaction.

We rebuild the usable and paired tickets from `fu/fu_ticket_last10.csv` and each ticket's group (chronic outliers, other top-5% machines, and all other machines) from the chronic outliers and the top 5% above. We check both against the table. For the paired tickets, we compute non-success codes per 1,000 transactions in each window. The rise is the median of the ticket windows' rates minus the median of the comparison windows' rates in each group. Windows with no transactions have no rate and are left out of the medians.

In [8]:
pre = R.read_csv(OUT, 'htm_pre_ticket.csv')  # one row per usable unplanned ticket

# The usable and paired tickets, rebuilt from the extract (unplanned is from the cell on the devices)
usable = unplanned[~unplanned.lookback_truncated & ~unplanned.after_telemetry_end]
comparison_windows = windows[(windows.window == 'control_minus35d') & windows.case_id.isin(usable.case_id)]
paired_ids = comparison_windows.case_id[comparison_windows.control_clean.eq(True)
                                        & ~comparison_windows.lookback_truncated]  # clean and complete windows
assert pre.case_id.is_unique and set(pre.case_id) == set(usable.case_id), 'htm_pre_ticket.csv is not the usable tickets'
assert (pre.control_clean_available == pre.case_id.isin(paired_ids)).all(), \
    'the paired tickets differ from htm_pre_ticket.csv'
# Each ticket's group, rebuilt from the chronic outliers and the top-5% machines' telemetry devices
top5_devices = set(machines.telemetry_id[machines.unplanned_tickets >= cut].dropna())
own_group = np.select([pre.machine_id.isin(chronic.machine_id), pre.machine_id.isin(top5_devices)],
                      ['chronic', 'workload_outsized'], 'other')
assert (own_group == pre.group).all(), "htm_pre_ticket.csv's groups differ from the chronic outliers and the top 5%"
paired = pre[pre.control_clean_available]


def per_1000(non_success, events):
    return (1000 * non_success / events).where(events > 0)


groups = {'chronic': 'chronic outliers', 'workload_outsized': 'other top-5% machines', 'other': 'all other machines'}
by_group = (paired.assign(ticket=per_1000(paired.non_success_events_30d, paired.all_events_30d),
                          comparison=per_1000(paired.control_non_success_events_30d, paired.control_all_events_30d))
            .groupby('group').agg(**{'paired tickets': ('case_id', 'size'), 'machines': ('machine_id', 'nunique'),
                                     'median, ticket window': ('ticket', 'median'),
                                     'median, comparison window': ('comparison', 'median')})
            .loc[list(groups)].rename(index=groups).rename_axis(None))
by_group['rise'] = by_group['median, ticket window'] - by_group['median, comparison window']
rise_chronic, rise_top5, rise_other = by_group['rise']
print(f'{len(pre):,} usable tickets, {len(paired):,} of them paired.')
print(f"The chronic outliers' rise ({rise_chronic:+.1f}) is no larger than the other groups' "
      f"({rise_top5:+.1f} and {rise_other:+.1f}): {rise_chronic <= min(rise_top5, rise_other)}")
by_group.round(1)

5,312 usable tickets, 3,259 of them paired.
The chronic outliers' rise (+20.3) is no larger than the other groups' (+21.6 and +21.5): True


,paired tickets,machines,"median, ticket window","median, comparison window",rise
chronic outliers,163,98,68.8,48.5,20.3
other top-5% machines,169,84,76.5,54.9,21.6
all other machines,2927,2322,62.9,41.4,21.5


## Check against the report

In [9]:
R.check([
    ('Chronic outliers', '113', R.num(n_chronic)),
    ('Customers with a chronic outlier', '82', R.num(n_chronic_customers)),
    ('Telemetry devices in the exposure population', '6,621', R.num(n_devices)),
    ("Top 5%'s share of the devices' tickets", '32.9%', R.pct(top_share, 1)),
    ('Same share by chance (median of 1,000 simulations)', '22.2%', R.pct(chance_median, 1)),
    ('Rate outliers before allowing for spread (docs/05)', '118', R.num(n_rate_outliers)),
    ('Rate outliers that still stand out under the negative binomial', '2', R.num(n_nb_kept)),
    ('Rise in non-success codes per 1,000: chronic outliers (Definitions)', '+20.3', f'{rise_chronic:+.1f}'),
    ('Rise: other top-5% machines (Definitions)', '+21.6', f'{rise_top5:+.1f}'),
    ('Rise: all other machines (Definitions)', '+21.5', f'{rise_other:+.1f}'),
    ("Chronic outliers' rise vs. other machines'", 'no more than',
     'no more than' if rise_chronic <= min(rise_top5, rise_other) else 'more than'),
]);

,Finding,Report,Recomputed,Match
0,Chronic outliers,113,113,yes
1,Customers with a chronic outlier,82,82,yes
2,Telemetry devices in the exposure population,"6,621","6,621",yes
3,Top 5%'s share of the devices' tickets,32.9%,32.9%,yes
4,"Same share by chance (median of 1,000 simulations)",22.2%,22.2%,yes
5,Rate outliers before allowing for spread (docs/05),118,118,yes
6,Rate outliers that still stand out under the negative binomial,2,2,yes
7,"Rise in non-success codes per 1,000: chronic outliers (Definitions)",+20.3,+20.3,yes
8,Rise: other top-5% machines (Definitions),+21.6,+21.6,yes
9,Rise: all other machines (Definitions),+21.5,+21.5,yes


All 11 values match the report.
